In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

base_path = os.path.dirname(os.getcwd())

# Machine-specific settings come from <project>/.env (see .env.example)
env = {}
with open(os.path.join(base_path, ".env")) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            env[key.strip()] = value.strip()

spark = (
    SparkSession.builder
    #.appName("sample-pyspark")
    .master("spark://localhost:7077")
    .config("spark.driver.host", env["SPARK_DRIVER_HOST"])  # Docker gateway: workers call back here
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.cores.max", "4")       # at most 4 cores in total
    .config("spark.executor.cores", "2")  # 2 executors with 2 cores each
    .config("spark.executor.memory", "2g")           # fits the worker's 2 GB
    .config("spark.eventLog.dir", env["SPARK_EVENTS_DIR"])
    .getOrCreate()
)
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/30 00:26:00 WARN Utils: Your hostname, <hostname>, resolves to a loopback address: 127.0.1.1; using <host-ip> instead (on interface wlo1)
26/10/30 00:26:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 00:26:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.0


In [2]:
orders = spark.read.csv(
    os.path.join(base_path, "data", "orders.csv"),
    header=True,
    inferSchema=True,
)
orders.printSchema()
orders.show(5)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- status: string (nullable = true)

+--------+-----------+----------+--------+-------------------+---------+
|order_id|customer_id|product_id|quantity|           order_ts|   status|
+--------+-----------+----------+--------+-------------------+---------+
|       1|          5|        17|       2|2024-06-25 12:54:24|delivered|
|       2|         83|        87|       2|2024-05-31 11:10:58|cancelled|
|       3|          5|        71|       1|2024-03-18 00:53:27|delivered|
|       4|          3|        75|       5|2023-08-11 20:27:33| returned|
|       5|         12|        57|       2|2023-09-08 07:09:39|  shipped|
+--------+-----------+----------+--------+-------------------+---------+
only showing top 5 rows


In [3]:
customers = spark.read.csv(
    os.path.join(base_path, "data", "customers.csv"),
    header=True,
    inferSchema=True,
)

customers.printSchema()
customers.show(5)

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- signup_date: date (nullable = true)

+-----------+--------------+--------------+-----+----------+-----------+
|customer_id|          name|          city|state|   segment|signup_date|
+-----------+--------------+--------------+-----+----------+-----------+
|          1|customer_00001|Rio de Janeiro|   RJ|enterprise| 2021-02-21|
|          2|customer_00002|  Porto Alegre|   RS|  consumer| 2022-05-17|
|          3|customer_00003|Belo Horizonte|   MG|  consumer| 2025-02-17|
|          4|customer_00004|      Brasília|   DF|enterprise| 2021-06-28|
|          5|customer_00005|        Recife|   PE|  consumer| 2021-03-07|
+-----------+--------------+--------------+-----+----------+-----------+
only showing top 5 rows


In [4]:
products = spark.read.csv(
    os.path.join(base_path, "data", "products.csv"),
    header=True,
    inferSchema=True,
)
products.printSchema()
products.show(5)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: double (nullable = true)

+----------+--------------------+-----------+----------+
|product_id|        product_name|   category|unit_price|
+----------+--------------------+-----------+----------+
|         1|   fashion_item_0001|    fashion|    408.81|
|         2|    sports_item_0002|     sports|     124.2|
|         3|    sports_item_0003|     sports|    758.98|
|         4|electronics_item_...|electronics|   3716.51|
|         5|     books_item_0005|      books|     49.94|
+----------+--------------------+-----------+----------+
only showing top 5 rows


In [5]:
all = customers.join(orders, customers.customer_id == orders.customer_id, "inner").join(products, orders.product_id == products.product_id, "inner")
all.createOrReplaceTempView("all")

In [7]:
spark.sql("SELECT * FROM all").show()

+-----------+--------------+--------------+-----+----------+-----------+--------+-----------+----------+--------+-------------------+---------+----------+--------------------+-----------+----------+
|customer_id|          name|          city|state|   segment|signup_date|order_id|customer_id|product_id|quantity|           order_ts|   status|product_id|        product_name|   category|unit_price|
+-----------+--------------+--------------+-----+----------+-----------+--------+-----------+----------+--------+-------------------+---------+----------+--------------------+-----------+----------+
|          1|customer_00001|Rio de Janeiro|   RJ|enterprise| 2021-02-21|      86|          1|        37|       3|2023-08-15 05:28:00|delivered|        37|electronics_item_...|electronics|   3535.19|
|          1|customer_00001|Rio de Janeiro|   RJ|enterprise| 2021-02-21|      85|          1|        58|       5|2023-05-02 13:58:46|delivered|        58|electronics_item_...|electronics|    439.69|
|    